In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import numpy as np

# --- Define Model Parameters ---
# These should match the shapes of your prepared data
HOUR_LENGTH = X_train.shape[1] if X_train.size > 0 else 24 # From your create_sequences function
num_zip_codes = X_train.shape[2] if X_train.size > 0 else (y_train.shape[1] if y_train.size > 0 else 3) # From your create_sequences function


# --- Shared Training Configuration ---
EPOCHS = 100 # You might need more or less depending on your data and patience
BATCH_SIZE = 32
LEARNING_RATE = 0.001

# Early Stopping to prevent overfitting and stop training when validation loss stops improving
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10, # Number of epochs with no improvement after which training will be stopped.
    restore_best_weights=True # Restores model weights from the epoch with the best value of the monitored quantity.
)

# Reduce learning rate when a metric has stopped improving
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5, # Factor by which the learning rate will be reduced. new_lr = lr * factor
    patience=5, # Number of epochs with no improvement after which learning rate will be reduced.
    min_lr=0.00001 # Lower bound on the learning rate.
)

callbacks_list = [early_stopping, reduce_lr]


print(f"Model Input Shape (hourlength, num_zip_codes): ({HOUR_LENGTH}, {num_zip_codes})")
print(f"Model Output Dimension (num_zip_codes): {num_zip_codes}")
print("-" * 50)


# --- 1. LSTM Model ---
print("Training LSTM Model...")
lstm_model = keras.Sequential([
    layers.Input(shape=(HOUR_LENGTH, num_zip_codes)),
    layers.LSTM(units=128, return_sequences=True), # Return sequences if stacking more LSTM layers
    layers.LSTM(units=64),
    layers.Dense(num_zip_codes)
])

lstm_model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                   loss='mse',
                   metrics=['mae', 'mse'])

lstm_model.summary()

if X_train.size > 0 and X_val.size > 0:
    history_lstm = lstm_model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val, y_val),
        callbacks=callbacks_list,
        verbose=1
    )
    print("\nEvaluating LSTM Model on Test Data:")
    lstm_results = lstm_model.evaluate(X_test, y_test, verbose=0)
    print(f"LSTM Test Loss (MSE): {lstm_results[0]:.4f}")
    print(f"LSTM Test MAE: {lstm_results[1]:.4f}")
    print(f"LSTM Test MSE: {lstm_results[2]:.4f}")
else:
    print("Skipping LSTM training: Training or Validation data is empty.")
print("-" * 50)




In [ ]:
# --- 2. TCN Model ---
print("\nTraining TCN Model...")

# Define a Temporal Block for TCN
def temporal_block(input_tensor, n_filters, kernel_size, dilation_rate, padding='causal', dropout_rate=0.2):
    conv1 = layers.Conv1D(
        filters=n_filters,
        kernel_size=kernel_size,
        dilation_rate=dilation_rate,
        padding=padding,
        activation='relu',
        kernel_initializer='he_normal'
    )(input_tensor)
    drop1 = layers.SpatialDropout1D(dropout_rate)(conv1) # SpatialDropout1D for time series
    conv2 = layers.Conv1D(
        filters=n_filters,
        kernel_size=kernel_size,
        dilation_rate=dilation_rate,
        padding=padding,
        activation='relu',
        kernel_initializer='he_normal'
    )(drop1)
    drop2 = layers.SpatialDropout1D(dropout_rate)(conv2)

    # Residual connection
    if input_tensor.shape[-1] != n_filters: # If input and output filter sizes don't match, use 1x1 conv
        res_connection = layers.Conv1D(filters=n_filters, kernel_size=1, padding='same')(input_tensor)
    else:
        res_connection = input_tensor

    return layers.Add()([res_connection, drop2])


tcn_input = keras.Input(shape=(HOUR_LENGTH, num_zip_codes))
x = tcn_input
n_filters = 64
kernel_size = 2
dropout_rate = 0.2

# Stack multiple temporal blocks with increasing dilation rates
for i in range(3): # Example with 3 blocks
    x = temporal_block(x, n_filters, kernel_size, dilation_rate=2**i, dropout_rate=dropout_rate)
    # n_filters can be increased for deeper layers, e.g., n_filters * 2 if desired

x = layers.GlobalAveragePooling1D()(x) # Or GlobalMaxPooling1D, or flatten if you want to use the last output
tcn_output = layers.Dense(num_zip_codes)(x)

tcn_model = keras.Model(inputs=tcn_input, outputs=tcn_output)

tcn_model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                  loss='mse',
                  metrics=['mae', 'mse'])

tcn_model.summary()

if X_train.size > 0 and X_val.size > 0:
    history_tcn = tcn_model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val, y_val),
        callbacks=callbacks_list,
        verbose=1
    )
    print("\nEvaluating TCN Model on Test Data:")
    tcn_results = tcn_model.evaluate(X_test, y_test, verbose=0)
    print(f"TCN Test Loss (MSE): {tcn_results[0]:.4f}")
    print(f"TCN Test MAE: {tcn_results[1]:.4f}")
    print(f"TCN Test MSE: {tcn_results[2]:.4f}")
else:
    print("Skipping TCN training: Training or Validation data is empty.")
print("-" * 50)

In [ ]:
# --- 3. Transformer Model ---
print("\nTraining Transformer Model...")

class MultiHeadSelfAttention(layers.Layer):
    def __init__(self, embed_dim, num_heads=8, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        if embed_dim % num_heads != 0:
            raise ValueError(
                f"embedding dimension = {embed_dim} should be divisible by number of heads = {num_heads}"
            )
        self.proj_dim = embed_dim // num_heads
        self.query_dense = layers.Dense(embed_dim)
        self.key_dense = layers.Dense(embed_dim)
        self.value_dense = layers.Dense(embed_dim)
        self.combine_heads = layers.Dense(embed_dim)

    def attention(self, query, key, value):
        score = tf.matmul(query, key, transpose_b=True)
        dim_key = tf.cast(tf.shape(key)[-1], tf.float32)
        scaled_score = score / tf.math.sqrt(dim_key)
        weights = tf.nn.softmax(scaled_score, axis=-1)
        output = tf.matmul(weights, value)
        return output, weights

    def separate_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.proj_dim))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        query = self.query_dense(inputs)
        key = self.key_dense(inputs)
        value = self.value_dense(inputs)

        query = self.separate_heads(query, batch_size)
        key = self.separate_heads(key, batch_size)
        value = self.separate_heads(value, batch_size)

        attention, weights = self.attention(query, key, value)
        attention = tf.transpose(attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(attention, (batch_size, -1, self.embed_dim))
        output = self.combine_heads(concat_attention)
        return output

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.att = MultiHeadSelfAttention(embed_dim, num_heads)
        self.ffn = keras.Sequential(
            [layers.Dense(ff_dim, activation="relu"), layers.Dense(embed_dim),]
        )
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# Positional Encoding for time series (can be learned or fixed)
class Time2Vec(layers.Layer):
    def __init__(self, output_dim, **kwargs):
        super(Time2Vec, self).__init__(**kwargs)
        self.output_dim = output_dim
        self.w = self.add_weight(name='weights', shape=(self.output_dim,),
                                 initializer='random_normal', trainable=True)
        self.p = self.add_weight(name='phases', shape=(self.output_dim,),
                                 initializer='random_normal', trainable=True)
        self.v = self.add_weight(name='v', shape=(self.output_dim,),
                                 initializer='random_normal', trainable=True)
        self.b = self.add_weight(name='b', shape=(self.output_dim,),
                                 initializer='random_normal', trainable=True)

    def call(self, x):
        # x is assumed to be a single feature (e.g., hour of day normalized, or timestep index)
        # For our case, we have `num_zip_codes` features at each timestep.
        # We need positional encoding per timestep.
        # A simpler approach for time series is to just add a learned embedding per timestep.
        # However, a proper Time2Vec or PositionalEmbedding would operate on the sequence length.
        # For this example, let's assume we're embedding the 'timestep' itself.
        # A fixed sinusoidal positional encoding is also an option.

        # Let's simplify and use a trainable positional embedding if the input is only the time steps
        # If the input is (batch, sequence_length, features), we need to apply positional encoding
        # to the sequence_length dimension and then add it to the features.
        # A common way is to make the input sequence length the position.
        # Since our X is already (samples, hourlength, num_zip_codes),
        # we need to embed the 'hourlength' dimension.
        # For simplicity, let's treat this as adding a learned embedding for each of the HOUR_LENGTH steps.

        # For a proper Transformer on time series, we need `embed_dim` which is usually the last dimension.
        # Here, `num_zip_codes` acts as our 'embed_dim' if we project the sequence.
        # Let's use num_zip_codes as embed_dim for simplicity for the TransformerBlock.
        # A better way is to project num_zip_codes into a higher embed_dim first.
        return x # Placeholder, will integrate directly into the model below

# Transformer model parameters
embed_dim = num_zip_codes # Embedding dimension for each token (here, each hour's zip code vector)
num_heads = 4 # Number of attention heads
ff_dim = 64   # Hidden layer size in feed forward network of transformer

transformer_input = keras.Input(shape=(HOUR_LENGTH, num_zip_codes))

# No separate Time2Vec layer here; instead, the TransformerBlock itself works on the sequence.
# If you need explicit positional encoding, you'd add it like:
# positions = tf.range(start=0, limit=HOUR_LENGTH, delta=1)
# positional_encoding = layers.Embedding(input_dim=HOUR_LENGTH, output_dim=embed_dim)(positions)
# x = transformer_input + positional_encoding # Shape mismatch if embed_dim != num_zip_codes

x = transformer_input
transformer_block1 = TransformerBlock(embed_dim, num_heads, ff_dim)
x = transformer_block1(x)
# Add more transformer blocks if desired
# x = TransformerBlock(embed_dim, num_heads, ff_dim)(x)

x = layers.GlobalAveragePooling1D()(x) # Pool across the time dimension
transformer_output = layers.Dense(num_zip_codes)(x)

transformer_model = keras.Model(inputs=transformer_input, outputs=transformer_output)

transformer_model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                          loss='mse',
                          metrics=['mae', 'mse'])

transformer_model.summary()

if X_train.size > 0 and X_val.size > 0:
    history_transformer = transformer_model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val, y_val),
        callbacks=callbacks_list,
        verbose=1
    )
    print("\nEvaluating Transformer Model on Test Data:")
    transformer_results = transformer_model.evaluate(X_test, y_test, verbose=0)
    print(f"Transformer Test Loss (MSE): {transformer_results[0]:.4f}")
    print(f"Transformer Test MAE: {transformer_results[1]:.4f}")
    print(f"Transformer Test MSE: {transformer_results[2]:.4f}")
else:
    print("Skipping Transformer training: Training or Validation data is empty.")
print("-" * 50)

In [ ]:
print("\nTraining Generic MLP Model for Time Series...")
mlp_model = keras.Sequential([
    layers.Input(shape=(HOUR_LENGTH, num_zip_codes)),
    layers.Flatten(), # Flatten the (hourlength, num_zip_codes) into a single vector
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_zip_codes) # Output layer for predicting all zip codes
])

mlp_model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                  loss='mse',
                  metrics=['mae', 'mse'])

mlp_model.summary()

if X_train.size > 0 and X_val.size > 0:
    history_mlp = mlp_model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val, y_val),
        callbacks=callbacks_list,
        verbose=1
    )
    print("\nEvaluating MLP Model on Test Data:")
    mlp_results = mlp_model.evaluate(X_test, y_test, verbose=0)
    print(f"MLP Test Loss (MSE): {mlp_results[0]:.4f}")
    print(f"MLP Test MAE: {mlp_results[1]:.4f}")
    print(f"MLP Test MSE: {mlp_results[2]:.4f}")
else:
    print("Skipping MLP training: Training or Validation data is empty.")
print("-" * 50)

print("\n--- Training Complete ---")
print("You can now inspect the history objects (e.g., `history_lstm.history`) for training/validation loss/metrics over epochs, and the `_results` variables for test set performance.")

In [ ]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 2.9 MB/s eta 0:00:00


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import numpy as np
import keras_tuner as kt



# --- Define Model Parameters ---
HOUR_LENGTH = X_train.shape[1] if X_train.size > 0 else 24
num_zip_codes = X_train.shape[2] if X_train.size > 0 else (y_train.shape[1] if y_train.size > 0 else 3)


# --- Shared Training Configuration ---
EPOCHS = 50 # Reduced for hyperparameter tuning to save time, adjust as needed
BATCH_SIZE = 32

# Early Stopping and ReduceLROnPlateau will be used within the tuner
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=0.00001
)

callbacks_list = [early_stopping, reduce_lr]

print(f"Model Input Shape (hourlength, num_zip_codes): ({HOUR_LENGTH}, {num_zip_codes})")
print(f"Model Output Dimension (num_zip_codes): {num_zip_codes}")
print("-" * 50)

# --- Hyperparameter Tuning with Keras Tuner ---

def build_model(hp):
    """
    Builds a Keras model for hyperparameter tuning.
    hp: HyperParameters object from Keras Tuner.
    """
    model = keras.Sequential()
    model.add(layers.Input(shape=(HOUR_LENGTH, num_zip_codes)))

    # Tune the number of units in the first LSTM layer
    hp_units_lstm1 = hp.Int('units_lstm1', min_value=32, max_value=256, step=32)
    model.add(layers.LSTM(units=hp_units_lstm1, return_sequences=True))

    # Tune the number of units in the second LSTM layer
    hp_units_lstm2 = hp.Int('units_lstm2', min_value=32, max_value=128, step=32)
    model.add(layers.LSTM(units=hp_units_lstm2))

    # Tune the activation function for the output layer (if applicable, 'linear' for regression often)
    # For regression, 'linear' is common, but you could explore 'relu' for positive-only outputs.
    hp_activation = hp.Choice('activation', values=['linear', 'relu'])
    model.add(layers.Dense(num_zip_codes, activation=hp_activation))

    # Tune the learning rate for the Adam optimizer
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                   loss='mse',
                   metrics=['mae', 'mse'])
    return model

# Initialize the Keras Tuner
# We'll use the RandomSearch tuner, but you could also use Hyperband or BayesianOptimization.
tuner = kt.RandomSearch(
    build_model,
    objective='val_loss', # Minimize validation loss
    max_trials=10,        # Number of different hyperparameter combinations to try
    executions_per_trial=1, # Number of models to train for each trial (for robustness)
    directory='keras_tuner_dir', # Directory to store results
    project_name='lstm_tuning' # Name of the tuning project
)

print("\nStarting Hyperparameter Search...")
# Start the hyperparameter search
tuner.search(X_train, y_train,
             epochs=EPOCHS,
             batch_size=BATCH_SIZE,
             validation_data=(X_val, y_val),
             callbacks=callbacks_list,
             verbose=1)

print("\n--- Hyperparameter Search Complete ---")

# Get the optimal hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
The optimal hyperparameters are:
Number of units in the first LSTM layer: {best_hps.get('units_lstm1')}
Number of units in the second LSTM layer: {best_hps.get('units_lstm2')}
Activation function for the output layer: {best_hps.get('activation')}
Learning rate for the optimizer: {best_hps.get('learning_rate')}
""")

# Build the best model found by the tuner
best_model = tuner.get_best_models(num_models=10)[0]

print("\nEvaluating the Best Model on Test Data:")
if X_test.size > 0:
    lstm_results = best_model.evaluate(X_test, y_test, verbose=0)
    print(f"Best Model Test Loss (MSE): {lstm_results[0]:.4f}")
    print(f"Best Model Test MAE: {lstm_results[1]:.4f}")
    print(f"Best Model Test MSE: {lstm_results[2]:.4f}")
else:
    print("Skipping evaluation: Test data is empty.")
print("-" * 50)

Trial 10 Complete [00h 01m 20s]
val_loss: 0.44470784068107605

Best val_loss So Far: 0.44381046295166016
Total elapsed time: 00h 14m 39s

--- Hyperparameter Search Complete ---

The optimal hyperparameters are:
Number of units in the first LSTM layer: 64
Number of units in the second LSTM layer: 64
Activation function for the output layer: linear
Learning rate for the optimizer: 0.0001



/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))



Evaluating the Best Model on Test Data:
Best Model Test Loss (MSE): 0.3953
Best Model Test MAE: 0.1708
Best Model Test MSE: 0.3953
--------------------------------------------------


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import numpy as np
import keras_tuner as kt




# --- Define Model Parameters ---
HOUR_LENGTH = X_train.shape[1] if X_train.size > 0 else 24
num_zip_codes = X_train.shape[2] if X_train.size > 0 else (y_train.shape[1] if y_train.size > 0 else 3)


# --- Shared Training Configuration ---
EPOCHS = 50 # Reduced for hyperparameter tuning to save time, adjust as needed
BATCH_SIZE = 32

# Early Stopping and ReduceLROnPlateau will be used within the tuner
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=0.00001
)

callbacks_list = [early_stopping, reduce_lr]

print(f"Model Input Shape (hourlength, num_zip_codes): ({HOUR_LENGTH}, {num_zip_codes})")
print(f"Model Output Dimension (num_zip_codes): {num_zip_codes}")
print("-" * 50)

# --- TCN Model with Hyperparameter Tuning ---

def temporal_block(input_tensor, n_filters, kernel_size, dilation_rate, padding='causal', dropout_rate=0.2):
    conv1 = layers.Conv1D(
        filters=n_filters,
        kernel_size=kernel_size,
        dilation_rate=dilation_rate,
        padding=padding,
        activation='relu',
        kernel_initializer='he_normal'
    )(input_tensor)
    drop1 = layers.SpatialDropout1D(dropout_rate)(conv1) # SpatialDropout1D for time series
    conv2 = layers.Conv1D(
        filters=n_filters,
        kernel_size=kernel_size,
        dilation_rate=dilation_rate,
        padding=padding,
        activation='relu',
        kernel_initializer='he_normal'
    )(drop1)
    drop2 = layers.SpatialDropout1D(dropout_rate)(conv2)

    # Residual connection
    if input_tensor.shape[-1] != n_filters: # If input and output filter sizes don't match, use 1x1 conv
        res_connection = layers.Conv1D(filters=n_filters, kernel_size=1, padding='same')(input_tensor)
    else:
        res_connection = input_tensor

    return layers.Add()([res_connection, drop2])


def build_tcn_model(hp):
    tcn_input = keras.Input(shape=(HOUR_LENGTH, num_zip_codes))
    x = tcn_input

    # Tune the number of filters in the temporal blocks
    hp_n_filters = hp.Int('n_filters', min_value=32, max_value=128, step=32)
    # Tune the kernel size in the temporal blocks
    hp_kernel_size = hp.Choice('kernel_size', values=[2, 3, 4])
    # Tune the dropout rate in the temporal blocks
    hp_dropout_rate = hp.Float('dropout_rate', min_value=0.0, max_value=0.5, step=0.1)


    # Stack multiple temporal blocks with increasing dilation rates
    for i in range(3): # Example with 3 blocks
        x = temporal_block(x, hp_n_filters, hp_kernel_size, dilation_rate=2**i, dropout_rate=hp_dropout_rate)

    x = layers.GlobalAveragePooling1D()(x) # Or GlobalMaxPooling1D, or flatten if you want to use the last output
    tcn_output = layers.Dense(num_zip_codes)(x)

    tcn_model = keras.Model(inputs=tcn_input, outputs=tcn_output)

    # Tune the learning rate for the Adam optimizer
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    tcn_model.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                      loss='mse',
                      metrics=['mae', 'mse'])
    return tcn_model

# Initialize the Keras Tuner
tuner = kt.RandomSearch(
    build_tcn_model,
    objective='val_loss',
    max_trials=10,
    executions_per_trial=1,
    directory='tcn_keras_tuner_dir',
    project_name='tcn_tuning'
)

print("\nStarting TCN Hyperparameter Search...")
# Start the hyperparameter search
tuner.search(X_train, y_train,
             epochs=EPOCHS,
             batch_size=BATCH_SIZE,
             validation_data=(X_val, y_val),
             callbacks=callbacks_list,
             verbose=1)

print("\n--- TCN Hyperparameter Search Complete ---")

# Get the optimal hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
The optimal TCN hyperparameters are:
Number of filters: {best_hps.get('n_filters')}
Kernel size: {best_hps.get('kernel_size')}
Dropout rate: {best_hps.get('dropout_rate')}
Learning rate: {best_hps.get('learning_rate')}
""")

# Build the best model found by the tuner
best_model = tuner.get_best_models(num_models=1)[0]


print("\nEvaluating the Best TCN Model on Test Data:")
if X_test.size > 0:
    tcn_results = best_model.evaluate(X_test, y_test, verbose=0)
    print(f"Best TCN Model Test Loss (MSE): {tcn_results[0]:.4f}")
    print(f"Best TCN Model Test MAE: {tcn_results[1]:.4f}")
    print(f"Best TCN Model Test MSE: {tcn_results[2]:.4f}")
else:
    print("Skipping evaluation: Test data is empty.")
print("-" * 50)

Trial 10 Complete [00h 01m 21s]
val_loss: 0.44888389110565186

Best val_loss So Far: 0.4451504349708557
Total elapsed time: 00h 13m 21s

--- TCN Hyperparameter Search Complete ---

The optimal TCN hyperparameters are:
Number of filters: 64
Kernel size: 3
Dropout rate: 0.30000000000000004
Learning rate: 0.001



/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 34 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))



Evaluating the Best TCN Model on Test Data:
Best TCN Model Test Loss (MSE): 0.3970
Best TCN Model Test MAE: 0.1731
Best TCN Model Test MSE: 0.3970
--------------------------------------------------


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import numpy as np
import keras_tuner as kt


# --- Define Model Parameters ---
HOUR_LENGTH = X_train.shape[1] if X_train.size > 0 else 24
num_zip_codes = X_train.shape[2] if X_train.size > 0 else (y_train.shape[1] if y_train.size > 0 else 3)


# --- Shared Training Configuration ---
EPOCHS = 50 # Reduced for hyperparameter tuning to save time, adjust as needed
BATCH_SIZE = 32

# Early Stopping and ReduceLROnPlateau will be used within the tuner
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=0.00001
)

callbacks_list = [early_stopping, reduce_lr]

print(f"Model Input Shape (hourlength, num_zip_codes): ({HOUR_LENGTH}, {num_zip_codes})")
print(f"Model Output Dimension (num_zip_codes): {num_zip_codes}")
print("-" * 50)


# --- Transformer Model with Hyperparameter Tuning ---

# MultiHeadSelfAttention and TransformerBlock classes remain the same as they define the architecture.
# We'll tune the parameters passed to their constructors.
class MultiHeadSelfAttention(layers.Layer):
    def __init__(self, embed_dim, num_heads=8, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        if embed_dim % num_heads != 0:
            raise ValueError(
                f"embedding dimension = {embed_dim} should be divisible by number of heads = {num_heads}"
            )
        self.proj_dim = embed_dim // num_heads
        self.query_dense = layers.Dense(embed_dim)
        self.key_dense = layers.Dense(embed_dim)
        self.value_dense = layers.Dense(embed_dim)
        self.combine_heads = layers.Dense(embed_dim)

    def attention(self, query, key, value):
        score = tf.matmul(query, key, transpose_b=True)
        dim_key = tf.cast(tf.shape(key)[-1], tf.float32)
        scaled_score = score / tf.math.sqrt(dim_key)
        weights = tf.nn.softmax(scaled_score, axis=-1)
        output = tf.matmul(weights, value)
        return output, weights

    def separate_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.proj_dim))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        query = self.query_dense(inputs)
        key = self.key_dense(inputs)
        value = self.value_dense(inputs)

        query = self.separate_heads(query, batch_size)
        key = self.separate_heads(key, batch_size)
        value = self.separate_heads(value, batch_size)

        attention, weights = self.attention(query, key, value)
        attention = tf.transpose(attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(attention, (batch_size, -1, self.embed_dim))
        output = self.combine_heads(concat_attention)
        return output

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.att = MultiHeadSelfAttention(embed_dim, num_heads)
        self.ffn = keras.Sequential(
            [layers.Dense(ff_dim, activation="relu"), layers.Dense(embed_dim),]
        )
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)


def build_transformer_model(hp):
    # Tune the embedding dimension for the Transformer
    # It's usually good to keep embed_dim divisible by num_heads
    hp_embed_dim = hp.Choice('embed_dim', values=[32, 64, 128])

    # Tune the number of attention heads
    hp_num_heads = hp.Choice('num_heads', values=[2, 4, 8])

    # Ensure embed_dim is divisible by num_heads for the chosen combination
    # This might require more advanced conditional logic or careful choice of `values`
    # For simplicity here, we assume valid combinations will be chosen or handled by `MultiHeadSelfAttention`.
    # A more robust way might be to limit the choices or implement a custom hyperparameter function.
    if hp_embed_dim % hp_num_heads != 0:
        # A simple way to handle invalid combinations is to return None, but Keras Tuner might not like it.
        # Alternatively, you can use a smaller embed_dim or larger num_heads
        # within the hp.Choice such that they are always compatible.
        # For this example, we'll proceed and let the MultiHeadSelfAttention raise an error if incompatible,
        # which will mark the trial as failed.
        pass

    # Tune the feed-forward network dimension
    hp_ff_dim = hp.Choice('ff_dim', values=[64, 128, 256])
    # Tune the dropout rate within the Transformer Block
    hp_dropout_rate = hp.Float('dropout_rate', min_value=0.1, max_value=0.4, step=0.1)

    # Tune the number of transformer blocks
    hp_num_transformer_blocks = hp.Int('num_transformer_blocks', min_value=1, max_value=3, step=1)

    transformer_input = keras.Input(shape=(HOUR_LENGTH, num_zip_codes))

    # Optional: Initial projection layer if num_zip_codes is very different from embed_dim
    # If num_zip_codes is small and embed_dim is large, an initial Dense layer can map features to embed_dim.
    x = layers.Dense(hp_embed_dim)(transformer_input) if num_zip_codes != hp_embed_dim else transformer_input

    # Add positional encoding (simple learned embedding for each time step)
    # This adds a trainable embedding based on the sequence length.
    positions = tf.range(start=0, limit=HOUR_LENGTH, delta=1)
    positional_embedding = layers.Embedding(input_dim=HOUR_LENGTH, output_dim=hp_embed_dim)(positions)
    # The positional embedding needs to be broadcasted or reshaped to match the batch dimension
    # Add expand_dims to make it (1, HOUR_LENGTH, embed_dim) so it broadcasts
    x = x + tf.expand_dims(positional_embedding, axis=0)


    for _ in range(hp_num_transformer_blocks):
        x = TransformerBlock(hp_embed_dim, hp_num_heads, hp_ff_dim, rate=hp_dropout_rate)(x)

    x = layers.GlobalAveragePooling1D()(x) # Pool across the time dimension

    # Output layer with a tunable activation if desired, but 'linear' is typical for regression.
    transformer_output = layers.Dense(num_zip_codes)(x)

    transformer_model = keras.Model(inputs=transformer_input, outputs=transformer_output)

    # Tune the learning rate for the Adam optimizer
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    transformer_model.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                              loss='mse',
                              metrics=['mae', 'mse'])
    return transformer_model

# Initialize the Keras Tuner
tuner = kt.RandomSearch(
    build_transformer_model,
    objective='val_loss',
    max_trials=10, # Number of different hyperparameter combinations to try
    executions_per_trial=1, # Number of models to train for each trial
    directory='transformer_keras_tuner_dir', # Directory to store results
    project_name='transformer_tuning' # Name of the tuning project
)

print("\nStarting Transformer Hyperparameter Search...")
# Start the hyperparameter search
tuner.search(X_train, y_train,
             epochs=EPOCHS,
             batch_size=BATCH_SIZE,
             validation_data=(X_val, y_val),
             callbacks=callbacks_list,
             verbose=1)

print("\n--- Transformer Hyperparameter Search Complete ---")

# Get the optimal hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
The optimal Transformer hyperparameters are:
Embedding Dimension: {best_hps.get('embed_dim')}
Number of Attention Heads: {best_hps.get('num_heads')}
Feed-Forward Dimension: {best_hps.get('ff_dim')}
Dropout Rate: {best_hps.get('dropout_rate')}
Number of Transformer Blocks: {best_hps.get('num_transformer_blocks')}
Learning Rate: {best_hps.get('learning_rate')}
""")

# Build the best model found by the tuner
best_model = tuner.get_best_models(num_models=1)[0]

print("\nEvaluating the Best Transformer Model on Test Data:")
if X_test.size > 0:
    transformer_results = best_model.evaluate(X_test, y_test, verbose=0)
    print(f"Best Transformer Model Test Loss (MSE): {transformer_results[0]:.4f}")
    print(f"Best Transformer Model Test MAE: {transformer_results[1]:.4f}")
    print(f"Best Transformer Model Test MSE: {transformer_results[2]:.4f}")
else:
    print("Skipping evaluation: Test data is empty.")
print("-" * 50)

Trial 10 Complete [00h 01m 14s]
val_loss: 0.44974246621131897

Best val_loss So Far: 0.4432264566421509
Total elapsed time: 00h 21m 10s

--- Transformer Hyperparameter Search Complete ---

The optimal Transformer hyperparameters are:
Embedding Dimension: 64
Number of Attention Heads: 4
Feed-Forward Dimension: 256
Dropout Rate: 0.30000000000000004
Number of Transformer Blocks: 3
Learning Rate: 0.01



/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 106 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))



Evaluating the Best Transformer Model on Test Data:
Best Transformer Model Test Loss (MSE): 0.3947
Best Transformer Model Test MAE: 0.1694
Best Transformer Model Test MSE: 0.3947
--------------------------------------------------


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import numpy as np
import keras_tuner as kt

# Assume X_train, y_train, X_val, y_val, X_test, y_test are already prepared
# For demonstration purposes, let's create some dummy data if they don't exist.
try:
    X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape, y_test.shape
except NameError:
    print("Dummy data created for demonstration. Replace with your actual data.")
    HOUR_LENGTH = 24
    NUM_ZIP_CODES = 3
    # Generating random data
    X_train = np.random.rand(100, HOUR_LENGTH, NUM_ZIP_CODES)
    y_train = np.random.rand(100, NUM_ZIP_CODES)
    X_val = np.random.rand(20, HOUR_LENGTH, NUM_ZIP_CODES)
    y_val = np.random.rand(20, NUM_ZIP_CODES)
    X_test = np.random.rand(20, HOUR_LENGTH, NUM_ZIP_CODES)
    y_test = np.random.rand(20, NUM_ZIP_CODES)


# --- Define Model Parameters ---
HOUR_LENGTH = X_train.shape[1] if X_train.size > 0 else 24
num_zip_codes = X_train.shape[2] if X_train.size > 0 else (y_train.shape[1] if y_train.size > 0 else 3)


# --- Shared Training Configuration ---
EPOCHS = 50 # Reduced for hyperparameter tuning to save time, adjust as needed
BATCH_SIZE = 32

# Early Stopping and ReduceLROnPlateau will be used within the tuner
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=0.00001
)

callbacks_list = [early_stopping, reduce_lr]

print(f"Model Input Shape (hourlength, num_zip_codes): ({HOUR_LENGTH}, {num_zip_codes})")
print(f"Model Output Dimension (num_zip_codes): {num_zip_codes}")
print("-" * 50)


# --- MLP Model with Hyperparameter Tuning ---

def build_mlp_model(hp):
    mlp_model = keras.Sequential()
    mlp_model.add(layers.Input(shape=(HOUR_LENGTH, num_zip_codes)))
    mlp_model.add(layers.Flatten()) # Flatten the (hourlength, num_zip_codes) into a single vector

    # Tune the number of hidden layers
    for i in range(hp.Int('num_hidden_layers', min_value=1, max_value=3)):
        # Tune the number of units in each dense layer
        mlp_model.add(layers.Dense(
            units=hp.Int(f'units_layer_{i}', min_value=32, max_value=256, step=32),
            activation='relu'
        ))
        # Tune the dropout rate after each hidden layer
        mlp_model.add(layers.Dropout(hp.Float(f'dropout_layer_{i}', min_value=0.1, max_value=0.5, step=0.1)))

    mlp_model.add(layers.Dense(num_zip_codes)) # Output layer

    # Tune the learning rate for the Adam optimizer
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    mlp_model.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                      loss='mse',
                      metrics=['mae', 'mse'])
    return mlp_model

# Initialize the Keras Tuner
tuner = kt.RandomSearch(
    build_mlp_model,
    objective='val_loss',
    max_trials=10,        # Number of different hyperparameter combinations to try
    executions_per_trial=1, # Number of models to train for each trial
    directory='mlp_keras_tuner_dir', # Directory to store results
    project_name='mlp_tuning' # Name of the tuning project
)

print("\nStarting MLP Hyperparameter Search...")
# Start the hyperparameter search
tuner.search(X_train, y_train,
             epochs=EPOCHS,
             batch_size=BATCH_SIZE,
             validation_data=(X_val, y_val),
             callbacks=callbacks_list,
             verbose=1)

print("\n--- MLP Hyperparameter Search Complete ---")

# Get the optimal hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# Print the optimal hyperparameters
print("\nThe optimal MLP hyperparameters are:")
print(f"Number of hidden layers: {best_hps.get('num_hidden_layers')}")
for i in range(best_hps.get('num_hidden_layers')):
    print(f"  Layer {i+1} Units: {best_hps.get(f'units_layer_{i}')}")
    print(f"  Layer {i+1} Dropout Rate: {best_hps.get(f'dropout_layer_{i}')}")
print(f"Learning Rate: {best_hps.get('learning_rate')}")


# Build the best model found by the tuner
best_model = tuner.get_best_models(num_models=1)[0]

print("\nEvaluating the Best MLP Model on Test Data:")
if X_test.size > 0:
    mlp_results = best_model.evaluate(X_test, y_test, verbose=0)
    print(f"Best MLP Model Test Loss (MSE): {mlp_results[0]:.4f}")
    print(f"Best MLP Model Test MAE: {mlp_results[1]:.4f}")
    print(f"Best MLP Model Test MSE: {mlp_results[2]:.4f}")
else:
    print("Skipping evaluation: Test data is empty.")
print("-" * 50)

print("\n--- Training Complete ---")
print("You can now inspect the history objects for training/validation loss/metrics over epochs, and the `_results` variables for test set performance.")

Trial 10 Complete [00h 00m 28s]
val_loss: 0.4438556134700775

Best val_loss So Far: 0.443032830953598
Total elapsed time: 00h 03m 02s

--- MLP Hyperparameter Search Complete ---

The optimal MLP hyperparameters are:
Number of hidden layers: 3
  Layer 1 Units: 96
  Layer 1 Dropout Rate: 0.4
  Layer 2 Units: 96
  Layer 2 Dropout Rate: 0.5
  Layer 3 Units: 64
  Layer 3 Dropout Rate: 0.5
Learning Rate: 0.01

Evaluating the Best MLP Model on Test Data:
Best MLP Model Test Loss (MSE): 0.3942
Best MLP Model Test MAE: 0.1689
Best MLP Model Test MSE: 0.3942
--------------------------------------------------

--- Training Complete ---
You can now inspect the history objects for training/validation loss/metrics over epochs, and the `_results` variables for test set performance.
